# PH3205 CT2 Exam Preparation — Master Code Sheet
## Ruturaj | All reusable building blocks

**Topics covered (from worksheets 4–9):**
- ODE methods: Euler, Midpoint, RK4, Verlet, Velocity Verlet, Leapfrog, Dormand-Prince
- Shooting method for BVPs and eigenvalue problems
- Crank-Nicolson for the heat equation
- FFT and TDSE (Strang splitting)
- Bisection and Secant root-finding (already known)

---
## PATTERN of CT1 questions (and expected CT2 pattern):
Each question typically has:
1. Analytical derivation / setting up equations
2. RK4 numerical integration
3. Bisection method to find specific times/positions
4. Plotting and comparing
5. Energy / phase portrait analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

---
# BLOCK 1: ROOT FINDING
(Needed in almost every CT question to find turning points, eigenvalues, etc.)

In [ ]:
# ============================================================
# BISECTION METHOD
# ============================================================
def bisection(f, a, b, tol=1e-6, maxiter=1000):
    """
    Find root of f in [a,b].
    Returns (root, iterations)
    """
    assert f(a)*f(b) < 0, "f(a) and f(b) must have opposite signs!"
    for i in range(maxiter):
        mid = (a + b) / 2.0
        if abs(b - a) < tol:
            return mid, i
        if f(a)*f(mid) < 0:
            b = mid
        else:
            a = mid
    return (a+b)/2.0, maxiter

# ============================================================
# SECANT METHOD
# ============================================================
def secant(f, x0, x1, tol=1e-6, maxiter=100):
    """
    Secant method for root finding.
    Returns (root, iterations)
    """
    for i in range(maxiter):
        f0, f1 = f(x0), f(x1)
        if abs(f1 - f0) < 1e-15:
            break
        x2 = x1 - f1*(x1 - x0)/(f1 - f0)
        if abs(x2 - x1) < tol:
            return x2, i
        x0, x1 = x1, x2
    return x1, maxiter

---
# BLOCK 2: ODE SOLVERS

**Key insight:** Always write your ODE as a first-order system:
```
dy/dt = f(t, y)   where y is a VECTOR
```
For a 2nd order ODE like x'' = g(x, x', t), write:
- y[0] = x (position)
- y[1] = x' (velocity)
- f = [y[1], g(y[0], y[1], t)]

In [ ]:
# ============================================================
# RK4 — SINGLE STEP
# ============================================================
def rk4_step(f, t, y, h):
    """
    Single RK4 step.
    f(t, y) -> dy/dt  (y is array)
    Returns y at t+h
    """
    k1 = h * f(t, y)
    k2 = h * f(t + h/2, y + k1/2)
    k3 = h * f(t + h/2, y + k2/2)
    k4 = h * f(t + h, y + k3)
    return y + (k1 + 2*k2 + 2*k3 + k4) / 6

# ============================================================
# RK4 — FULL INTEGRATION
# ============================================================
def rk4_integrate(f, t0, tf, y0, h):
    """
    Integrate ODE from t0 to tf using RK4.
    Returns arrays: t_arr, y_arr
    y_arr[i] is the state at time t_arr[i]
    """
    t_arr = [t0]
    y_arr = [np.array(y0, dtype=float)]
    t = t0
    y = np.array(y0, dtype=float)
    while t < tf - 1e-12:
        h_use = min(h, tf - t)  # don't overshoot
        y = rk4_step(f, t, y, h_use)
        t += h_use
        t_arr.append(t)
        y_arr.append(y.copy())
    return np.array(t_arr), np.array(y_arr)

# ============================================================
# EULER — FULL INTEGRATION
# ============================================================
def euler_integrate(f, t0, tf, y0, h):
    t_arr = [t0]
    y_arr = [np.array(y0, dtype=float)]
    t = t0
    y = np.array(y0, dtype=float)
    while t < tf - 1e-12:
        h_use = min(h, tf - t)
        y = y + h_use * f(t, y)
        t += h_use
        t_arr.append(t)
        y_arr.append(y.copy())
    return np.array(t_arr), np.array(y_arr)

# ============================================================
# MIDPOINT (RK2) — FULL INTEGRATION
# ============================================================
def midpoint_integrate(f, t0, tf, y0, h):
    t_arr = [t0]
    y_arr = [np.array(y0, dtype=float)]
    t = t0
    y = np.array(y0, dtype=float)
    while t < tf - 1e-12:
        h_use = min(h, tf - t)
        k1 = h_use * f(t, y)
        y = y + h_use * f(t + h_use/2, y + k1/2)
        t += h_use
        t_arr.append(t)
        y_arr.append(y.copy())
    return np.array(t_arr), np.array(y_arr)

# ============================================================
# VELOCITY VERLET
# For x'' = a(x), i.e., acceleration depends only on position
# ============================================================
def vel_verlet_integrate(accel, t0, tf, x0, v0, h):
    """
    accel(x) -> acceleration at position x
    Returns t_arr, x_arr, v_arr
    """
    t_arr = [t0]
    x_arr = [x0]
    v_arr = [v0]
    t, x, v = t0, x0, v0
    a = accel(x)
    while t < tf - 1e-12:
        h_use = min(h, tf - t)
        x_new = x + v*h_use + 0.5*a*h_use**2
        a_new = accel(x_new)
        v_new = v + 0.5*(a + a_new)*h_use
        t += h_use
        x, v, a = x_new, v_new, a_new
        t_arr.append(t)
        x_arr.append(x)
        v_arr.append(v)
    return np.array(t_arr), np.array(x_arr), np.array(v_arr)

---
# BLOCK 3: SHOOTING METHOD FOR EIGENVALUES

**Pattern:** Find energy E such that the wavefunction satisfies boundary conditions.

**Key equations:**
- TISE in natural units (ℏ=1, 2m=1): ψ'' = (V(x) - E)ψ
- Written as 1st-order system: [ψ, ψ'] -> [ψ', (V-E)ψ]
- Score function: ψ(x_right) should be 0 for bound states
  (or log-derivative matching for symmetric potentials)

In [ ]:
# ============================================================
# GENERIC SHOOTING METHOD FOR TISE
# Works for: particle-in-a-box-like problems, finite wells, etc.
# ============================================================

def shoot_1D(E, V_func, x_left, x_right, N=1000):
    """
    Integrate TISE from left to right for trial energy E.
    ψ'' = (V(x) - E)ψ
    Returns x_arr, psi_arr
    Boundary: ψ(x_left) = 0, ψ'(x_left) = 1 (trial slope)
    """
    h = (x_right - x_left) / N
    
    def f_tise(x, y):
        # y[0] = psi, y[1] = psi'
        return np.array([y[1], (V_func(x) - E) * y[0]])
    
    x = x_left
    y = np.array([0.0, 1.0])  # psi=0, psi'=1 at left wall
    x_arr = [x]
    psi_arr = [y[0]]
    
    for _ in range(N):
        y = rk4_step(f_tise, x, y, h)
        x += h
        x_arr.append(x)
        psi_arr.append(y[0])
    
    return np.array(x_arr), np.array(psi_arr)

def score_psi_right(E, V_func, x_left, x_right, N=1000):
    """Returns ψ at the right boundary — eigenvalue when this is 0"""
    _, psi = shoot_1D(E, V_func, x_left, x_right, N)
    return psi[-1]

# ============================================================
# LOG-DERIVATIVE MATCHING (for symmetric potentials — CT1 Q4 style)
# Shoot from BOTH sides to matching point x_m
# ============================================================

def shoot_from_left(E, V_func, x_start, x_match, N=500):
    """Integrate rightward from x_start to x_match."""
    kappa = np.sqrt(max(-E, 1e-15))  # decay constant outside well
    h = (x_match - x_start) / N
    
    def f_tise(x, y):
        return np.array([y[1], (V_func(x) - E) * y[0]])
    
    # Asymptotic decaying initial condition
    psi0 = np.exp(-kappa * abs(x_start))
    dpsi0 = kappa * psi0  # derivative of e^{-kappa*|x|} at x<0 is +kappa * e^{...}
    y = np.array([psi0, dpsi0])
    x = x_start
    
    for _ in range(N):
        y = rk4_step(f_tise, x, y, h)
        x += h
    return y  # [psi, psi'] at x_match

def shoot_from_right(E, V_func, x_end, x_match, N=500):
    """Integrate leftward from x_end to x_match."""
    kappa = np.sqrt(max(-E, 1e-15))
    h = (x_match - x_end) / N  # negative h since going left
    
    def f_tise(x, y):
        return np.array([y[1], (V_func(x) - E) * y[0]])
    
    psi0 = np.exp(-kappa * abs(x_end))
    dpsi0 = -kappa * psi0  # derivative of e^{-kappa*x} at x>0 is -kappa
    y = np.array([psi0, dpsi0])
    x = x_end
    
    for _ in range(N):
        y = rk4_step(f_tise, x, y, h)  # h is negative
        x += h
    return y  # [psi, psi'] at x_match

def mismatch_F(E, V_func, x_left=-3.0, x_right=3.0, x_match=0.5, N=500):
    """
    Mismatch function F(E) = psi'_L/psi_L - psi'_R/psi_R at x_match.
    Eigenvalue when F(E) = 0.
    """
    yL = shoot_from_left(E, V_func, x_left, x_match, N)
    yR = shoot_from_right(E, V_func, x_right, x_match, N)
    
    if abs(yL[0]) < 1e-14 or abs(yR[0]) < 1e-14:
        return 0.0  # degenerate case
    
    return yL[1]/yL[0] - yR[1]/yR[0]

def find_eigenvalues(V_func, V0, n_grid=500, x_left=-3.0, x_right=3.0, x_match=0.5):
    """
    Scan over E in [-V0, 0) and find all zeros of F(E).
    Returns list of eigenvalues.
    """
    E_grid = np.linspace(-V0 + 0.01, -0.01, n_grid)
    F_vals = np.array([mismatch_F(E, V_func, x_left, x_right, x_match) for E in E_grid])
    
    eigenvalues = []
    for i in range(len(F_vals)-1):
        if F_vals[i]*F_vals[i+1] < 0 and abs(F_vals[i]) < 50 and abs(F_vals[i+1]) < 50:
            # Sign change found — use bisection
            f_wrap = lambda E: mismatch_F(E, V_func, x_left, x_right, x_match)
            root, _ = bisection(f_wrap, E_grid[i], E_grid[i+1], tol=1e-6)
            eigenvalues.append(root)
    
    return eigenvalues

def reconstruct_eigenfunction(E, V_func, x_left=-3.0, x_right=3.0, x_match=0.5, N=1000):
    """
    Reconstruct the full eigenfunction by shooting from both sides and joining at x_match.
    Returns x_arr, psi_arr (normalized).
    """
    def f_tise(x, y):
        return np.array([y[1], (V_func(x) - E) * y[0]])
    
    kappa = np.sqrt(max(-E, 1e-15))
    
    # Left side
    h = (x_match - x_left) / N
    psi0 = np.exp(-kappa * abs(x_left))
    yL = np.array([psi0, kappa*psi0])
    x = x_left
    xL_arr, psiL_arr = [x], [yL[0]]
    for _ in range(N):
        yL = rk4_step(f_tise, x, yL, h)
        x += h
        xL_arr.append(x)
        psiL_arr.append(yL[0])
    
    # Right side
    h2 = (x_match - x_right) / N  # negative
    psiR0 = np.exp(-kappa * abs(x_right))
    yR = np.array([psiR0, -kappa*psiR0])
    x = x_right
    xR_arr, psiR_arr = [x], [yR[0]]
    for _ in range(N):
        yR = rk4_step(f_tise, x, yR, h2)
        x += h2
        xR_arr.append(x)
        psiR_arr.append(yR[0])
    
    # Scale right so psi_L(x_match) = psi_R(x_match)
    scale = psiL_arr[-1] / psiR_arr[-1]
    psiR_arr = [p * scale for p in psiR_arr]
    
    # Join: left goes x_left -> x_match, right goes x_right -> x_match (reverse it)
    x_full = np.array(xL_arr + list(reversed(xR_arr[1:])))
    psi_full = np.array(psiL_arr + list(reversed(psiR_arr[1:])))
    
    # Normalize using trapezoidal rule
    norm = np.trapz(psi_full**2, x_full)
    psi_full /= np.sqrt(norm)
    
    return x_full, psi_full

---
# BLOCK 4: HEAT EQUATION (Crank-Nicolson)

**PDE:** ∂u/∂t = α ∂²u/∂x²

**Crank-Nicolson update:**
A·u^{n+1} = B·u^n + boundary terms

where r = α·Δt/Δx², and:
- A: tridiagonal with [-r/2, 1+r, -r/2]
- B: tridiagonal with [r/2, 1-r, r/2]

In [ ]:
# ============================================================
# THOMAS ALGORITHM (tridiagonal solver)
# ============================================================
def thomas(a, b, c, d):
    """
    Solve tridiagonal system: a[i]*x[i-1] + b[i]*x[i] + c[i]*x[i+1] = d[i]
    a: sub-diagonal (a[0] unused)
    b: main diagonal
    c: super-diagonal (c[n-1] unused)
    Returns solution x.
    IMPORTANT: pass copies of arrays — this modifies them!
    """
    a, b, c, d = a.copy(), b.copy(), c.copy(), d.copy()
    n = len(d)
    # Forward sweep
    for i in range(1, n):
        m = a[i] / b[i-1]
        b[i] -= m * c[i-1]
        d[i] -= m * d[i-1]
    # Back substitution
    x = np.zeros(n)
    x[-1] = d[-1] / b[-1]
    for i in range(n-2, -1, -1):
        x[i] = (d[i] - c[i]*x[i+1]) / b[i]
    return x

# ============================================================
# CRANK-NICOLSON HEAT EQUATION SOLVER
# ============================================================
def crank_nicolson(u0, x_arr, t_final, dt, alpha, T_left, T_right):
    """
    Solve heat equation with Dirichlet BCs.
    u0: initial temperature array (including boundary points)
    x_arr: spatial grid
    Returns: t_snapshots, u_snapshots (list of arrays at various times)
    """
    N = len(u0) - 2  # interior points
    dx = x_arr[1] - x_arr[0]
    r = alpha * dt / dx**2
    
    # Build matrices for interior points
    # A * u^{n+1} = B * u^n + rhs_boundary
    a_diag = np.full(N, -r/2)   # sub-diagonal
    b_diag = np.full(N, 1 + r)  # main diagonal
    c_diag = np.full(N, -r/2)   # super-diagonal
    
    u = u0.copy()
    t = 0.0
    snapshots = [(t, u.copy())]
    
    n_steps = int(t_final / dt)
    save_every = max(1, n_steps // 10)
    
    for step in range(n_steps):
        u_int = u[1:-1]  # interior
        
        # Right-hand side from B matrix
        rhs = np.zeros(N)
        rhs[0] = (r/2)*u[0] + (1-r)*u_int[0] + (r/2)*u_int[1]
        rhs[-1] = (r/2)*u_int[-2] + (1-r)*u_int[-1] + (r/2)*u[-1]
        for i in range(1, N-1):
            rhs[i] = (r/2)*u_int[i-1] + (1-r)*u_int[i] + (r/2)*u_int[i+1]
        
        # Add boundary contributions to A side
        rhs[0]  += (r/2) * T_left
        rhs[-1] += (r/2) * T_right
        
        # Solve
        u_new_int = thomas(a_diag, b_diag, c_diag, rhs)
        u[1:-1] = u_new_int
        u[0] = T_left
        u[-1] = T_right
        t += dt
        
        if step % save_every == 0:
            snapshots.append((t, u.copy()))
    
    snapshots.append((t, u.copy()))
    return snapshots

---
# BLOCK 5: TDSE (Strang Splitting + FFT)

**Split into:**
1. Half-step potential: ψ → ψ·exp(-i·V·Δt/2)
2. Full-step kinetic (in k-space): ψ̂ → ψ̂·exp(-i·k²·Δt/2)
3. Half-step potential again

**Key: initial wavepacket MUST have momentum term exp(ik₀x)**

In [ ]:
# ============================================================
# TDSE SOLVER via Strang Splitting
# ============================================================
def tdse_strang(psi0, x_arr, V_arr, dt, n_steps, save_every=10):
    """
    Solve TDSE: iψ_t = (-1/2 ∂²/∂x² + V)ψ  [ℏ=1, m=1 units]
    Using Strang splitting.
    
    psi0: initial wavefunction (complex array)
    x_arr: spatial grid (uniform)
    V_arr: potential on same grid
    Returns: list of (t, |ψ|²) snapshots
    """
    N = len(x_arr)
    dx = x_arr[1] - x_arr[0]
    
    # k-space grid (for FFT)
    dk = 2*np.pi / (N*dx)
    k = np.fft.fftfreq(N) * N * dk  # wavenumber array
    
    # Propagators
    exp_V_half = np.exp(-1j * V_arr * dt / 2)
    exp_T_full = np.exp(-1j * k**2 * dt / 2)  # kinetic: -∂²/2 → k²/2 in k-space
    
    psi = psi0.copy().astype(complex)
    snapshots = [(0.0, np.abs(psi)**2)]
    
    for step in range(n_steps):
        # Half-step potential
        psi *= exp_V_half
        # Full-step kinetic (in k-space)
        psi_k = np.fft.fft(psi)
        psi_k *= exp_T_full
        psi = np.fft.ifft(psi_k)
        # Half-step potential
        psi *= exp_V_half
        
        if step % save_every == 0:
            snapshots.append(((step+1)*dt, np.abs(psi)**2))
    
    return snapshots

def gaussian_wavepacket(x, x0, sigma, k0):
    """
    Normalized Gaussian wavepacket centered at x0 with width sigma and momentum k0.
    MUST include exp(ik0*x) for it to travel!
    """
    psi = (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-(x-x0)**2 / (2*sigma**2)) * np.exp(1j*k0*x)
    # Normalize numerically
    norm = np.trapz(np.abs(psi)**2, x)
    return psi / np.sqrt(norm)

---
# BLOCK 6: NUMERICAL INTEGRATION
(For computing depths, energies, normalizations)

In [ ]:
def trapz(f_arr, x_arr):
    """Trapezoidal integration (or just use np.trapz)"""
    return np.trapz(f_arr, x_arr)

def simpsons(f_arr, h):
    """
    Simpson's 1/3 rule. N must be even (len(f_arr) = N+1 must be odd).
    """
    N = len(f_arr) - 1
    assert N % 2 == 0, "Need even N for Simpson's 1/3"
    result = f_arr[0] + f_arr[-1]
    for i in range(1, N):
        result += (4 if i%2==1 else 2) * f_arr[i]
    return result * h / 3

---
# PREDICTED CT2 QUESTION TYPES

Based on the CT1 pattern, here are the most likely question templates:

## Q1: Physical dynamics with drag (like rocket/Stokes)
- Setup 1st-order system
- RK4 integration, phase portrait
- Bisection to find specific times
- Numerical integration for position from velocity

## Q2: Oscillator / damped system
- Could be: Van der Pol, projectile, damped Gaussian hills (like CT1 Q3)
- Phase portrait spiraling inward
- Finding turning points using bisection on v(t)
- Energy analysis

## Q3: Quantum BVP / Eigenvalues
- TISE shooting method
- Log-derivative matching (CT1 Q4 style)
- Plot eigenfunctions offset by eigenvalues
- Count nodes

## Possibly: Heat equation (WS8) or TDSE (WS9)
- Crank-Nicolson setup
- Strang splitting TDSE
---

# WORKED EXAMPLE 1: Damped Oscillator in Double-Well
## (Same structure as CT1 Q3 — most likely to reappear in CT2)

**Potential:** V(x) = V1·exp(-(x+b)²/2σ1²) + V2·exp(-(x-b)²/2σ2²)

**EOM:** mẍ = -dV/dx - γẋ

**First-order system:**
- ẋ = v
- v̇ = (1/m)·[-dV/dx - γv]

In [ ]:
# ============================================================
# DOUBLE GAUSSIAN WELL — FULL SOLUTION
# ============================================================

# Parameters (CT1 Q3 values — change as needed)
m  = 1.0
V1 = 3.0;  sigma1 = 0.5
V2 = 5.0;  sigma2 = 0.4
b  = 1.0
gamma = 0.3
x0, v0 = -0.5, 2.0

def V_gauss(x):
    return V1*np.exp(-(x+b)**2/(2*sigma1**2)) + V2*np.exp(-(x-b)**2/(2*sigma2**2))

def dV_dx(x):
    t1 = V1*(x+b)/sigma1**2 * np.exp(-(x+b)**2/(2*sigma1**2))
    t2 = V2*(x-b)/sigma2**2 * np.exp(-(x-b)**2/(2*sigma2**2))
    return t1 + t2

def f_double_well(t, y):
    x, v = y
    dvdt = (1/m) * (-dV_dx(x) - gamma*v)
    return np.array([v, dvdt])

# Integrate
t_arr, y_arr = rk4_integrate(f_double_well, 0, 30, [x0, v0], h=1e-3)
x_arr_dw = y_arr[:,0]
v_arr_dw = y_arr[:,1]

# ---- Plot potential landscape ----
fig, axes = plt.subplots(1, 3, figsize=(15,4))

x_plot = np.linspace(-3, 3, 500)
E0 = 0.5*m*v0**2 + V_gauss(x0)

axes[0].plot(x_plot, V_gauss(x_plot), 'b-', label='V(x)')
axes[0].axhline(E0, color='r', linestyle='--', label=f'E0={E0:.3f}')
axes[0].set_title('Potential Landscape')
axes[0].set_xlabel('x'); axes[0].legend(); axes[0].grid()

# ---- Plot v(t) ----
axes[1].plot(t_arr, v_arr_dw)
axes[1].axhline(0, color='k', linestyle='--')
axes[1].set_title('v(t)')
axes[1].set_xlabel('t'); axes[1].set_ylabel('v'); axes[1].grid()

# ---- Phase portrait ----
axes[2].plot(x_arr_dw, v_arr_dw)
axes[2].set_title('Phase Portrait v vs x')
axes[2].set_xlabel('x'); axes[2].set_ylabel('v'); axes[2].grid()

plt.tight_layout()
plt.show()

# ---- Find turning points on RIGHT wall using bisection ----
def v_at_t(t_target):
    """Interpolate v(t) at a given time target."""
    idx = np.searchsorted(t_arr, t_target)
    if idx == 0: return v_arr_dw[0]
    if idx >= len(t_arr): return v_arr_dw[-1]
    # Linear interpolation
    frac = (t_target - t_arr[idx-1]) / (t_arr[idx] - t_arr[idx-1])
    return v_arr_dw[idx-1] + frac*(v_arr_dw[idx] - v_arr_dw[idx-1])

def x_at_t(t_target):
    idx = np.searchsorted(t_arr, t_target)
    if idx == 0: return x_arr_dw[0]
    if idx >= len(t_arr): return x_arr_dw[-1]
    frac = (t_target - t_arr[idx-1]) / (t_arr[idx] - t_arr[idx-1])
    return x_arr_dw[idx-1] + frac*(x_arr_dw[idx] - x_arr_dw[idx-1])

# Find intervals where v changes sign AND x > 0 (right wall turning points)
right_wall_brackets = []
for i in range(len(v_arr_dw)-1):
    if v_arr_dw[i]*v_arr_dw[i+1] < 0 and x_arr_dw[i] > 0:
        right_wall_brackets.append((t_arr[i], t_arr[i+1]))

print(f"Found {len(right_wall_brackets)} turning points on right wall")
turning_times = []
for ta, tb in right_wall_brackets[:3]:  # first 3
    t_star, _ = bisection(v_at_t, ta, tb, tol=1e-6)
    x_star = x_at_t(t_star)
    E_star = 0.5*m*v_at_t(t_star)**2 + V_gauss(x_star)
    turning_times.append(t_star)
    print(f"  t* = {t_star:.6f}, x* = {x_star:.6f}, E = {E_star:.6f}")

---
# WORKED EXAMPLE 2: TISE Quantum Shooting (CT1 Q4 style)

**Elliptic potential:**
V(x) = -V0·√(a²-x²) for |x| ≤ a, else 0

(Or any well — just change V_func)

In [ ]:
# ============================================================
# ELLIPTIC WELL EIGENVALUES (CT1 Q4)
# ============================================================
V0_qm = 10.0
a_qm  = 1.0

def V_elliptic(x):
    if abs(x) <= a_qm:
        return -V0_qm * np.sqrt(a_qm**2 - x**2)
    return 0.0

# Vectorized version
def V_elliptic_vec(x):
    return np.where(np.abs(x) <= a_qm, -V0_qm * np.sqrt(np.maximum(a_qm**2 - x**2, 0)), 0.0)

# Find eigenvalues
eigenvalues = find_eigenvalues(V_elliptic, V0_qm, n_grid=1000, 
                                x_left=-3.0, x_right=3.0, x_match=0.5)
print("Eigenvalues found:")
for n, E in enumerate(eigenvalues):
    print(f"  E_{n} = {E:.6f}")

# Plot eigenfunctions
x_plot = np.linspace(-3, 3, 500)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_plot, V_elliptic_vec(x_plot), 'k-', lw=2, label='V(x)')

c = 1.5  # scale factor for visibility
for n, E in enumerate(eigenvalues):
    x_ef, psi_ef = reconstruct_eigenfunction(E, V_elliptic, 
                                               x_left=-3.0, x_right=3.0, x_match=0.5)
    nodes = np.sum(psi_ef[:-1]*psi_ef[1:] < 0)
    ax.plot(x_ef, E + c*psi_ef, label=f'E_{n}={E:.3f}, nodes={nodes}')

ax.set_xlabel('x'); ax.set_ylabel('Energy / ψ')
ax.set_title('Eigenfunctions of Elliptic Well')
ax.legend(); ax.grid()
plt.tight_layout()
plt.show()

---
# WORKED EXAMPLE 3: Stokes' Law / Terminal Velocity (CT1 Q2 style)

**EOM:** dv/dt = (vT - v)/τ

**Analytical:** v(t) = vT(1 - e^{-t/τ})

**t* = τ ln(100)** when v = 0.99 vT

**x(t) = vT·[t + τ·e^{-t/τ} - τ]**

In [ ]:
# ============================================================
# STOKES' LAW — FULL SOLUTION
# ============================================================
r_s  = 1.5e-3    # sphere radius (m)
rho_s = 7800.0   # sphere density (kg/m^3)
rho_f = 1260.0   # fluid density (kg/m^3)
eta  = 1.49      # dynamic viscosity (Pa·s)
g    = 9.81      # m/s^2

m_sphere = (4/3) * np.pi * r_s**3 * rho_s
F_buoy   = (4/3) * np.pi * r_s**3 * rho_f * g
F_grav   = m_sphere * g
drag_coef = 6 * np.pi * eta * r_s  # Stokes drag: F_drag = drag_coef * v

# Terminal velocity and relaxation time
vT  = (F_grav - F_buoy) / drag_coef
tau = m_sphere / drag_coef

print(f"Terminal velocity vT = {vT:.4f} m/s")
print(f"Relaxation time  tau = {tau:.4f} s")

# Analytical solution
t_star_analytic = tau * np.log(100)
print(f"t* (99% of vT)   = {t_star_analytic:.4f} s")

# x(t) = vT*(t + tau*exp(-t/tau) - tau)
x_star_analytic = vT * (t_star_analytic + tau*np.exp(-t_star_analytic/tau) - tau)
print(f"x* (depth at t*) = {x_star_analytic:.4f} m")

# RK4 solution
def f_stokes(t, y):
    v = y[0]
    return np.array([(vT - v) / tau])

t_arr_s, y_arr_s = rk4_integrate(f_stokes, 0, 3*t_star_analytic, [0.0], h=1e-3)
v_arr_s = y_arr_s[:,0]

# Analytical
v_exact = vT * (1 - np.exp(-t_arr_s/tau))

plt.figure(figsize=(8,4))
plt.plot(t_arr_s, v_arr_s, 'b-', label='RK4')
plt.plot(t_arr_s, v_exact, 'r--', label='Analytical')
plt.axhline(0.99*vT, color='g', linestyle=':', label='0.99 vT')
plt.xlabel('t (s)'); plt.ylabel('v (m/s)')
plt.title("Stokes' Law: Sphere in Viscous Fluid")
plt.legend(); plt.grid(); plt.tight_layout(); plt.show()

# Bisection to find t* numerically
def v_interp(t_target):
    idx = np.searchsorted(t_arr_s, t_target)
    if idx == 0: return v_arr_s[0]
    frac = (t_target - t_arr_s[idx-1])/(t_arr_s[idx]-t_arr_s[idx-1])
    return v_arr_s[idx-1] + frac*(v_arr_s[idx]-v_arr_s[idx-1])

t_star_num, _ = bisection(lambda t: v_interp(t) - 0.99*vT, 0, t_arr_s[-1], tol=1e-4)
print(f"\nt* numerical = {t_star_num:.4f} s")
print(f"t* analytic  = {t_star_analytic:.4f} s")

# Numerical depth by trapezoidal integration
mask = t_arr_s <= t_star_num
x_star_num = np.trapz(v_arr_s[mask], t_arr_s[mask])
print(f"x* numerical = {x_star_num:.4f} m")
print(f"x* analytic  = {x_star_analytic:.4f} m")

---
# WORKED EXAMPLE 4: Rocket Problem (CT1 Q1 style)

**Phase 1 (0 ≤ t ≤ T):** (2m0 - αt)·dv/dt = vrel·α - γv

**Phase 2 (T < t ≤ 2T):** m0·dv/dt = -γv

**Analytical Phase 1 (γ ≠ 0):**
v(t) = (vrel·α/γ)·[1 - (1 - t/(2T))^{γ/α}]

**For γ = 0:** v(t) = vrel·ln(2m0/(2m0-αt))

In [ ]:
# ============================================================
# ROCKET DYNAMICS — RK4 with phase switching
# ============================================================
m0   = 1.0
alpha = 1.0   # fuel burn rate
vrel = 1.0    # exhaust speed
T    = m0 / alpha  # time to exhaust fuel (T = m0/alpha = 1)

for gamma in [0.0, alpha/2, alpha]:
    def f_rocket(t, y):
        v = y[0]
        if t <= T:
            mass = 2*m0 - alpha*t
            dvdt = (vrel*alpha - gamma*v) / mass
        else:
            dvdt = -gamma*v / m0
        return np.array([dvdt])
    
    t_arr_r, y_arr_r = rk4_integrate(f_rocket, 0, 2*T, [0.0], h=1e-3)
    v_arr_r = y_arr_r[:,0]
    
    # Analytical Phase 1
    t1 = t_arr_r[t_arr_r <= T]
    if gamma == 0:
        v_anal1 = vrel * np.log(2*m0/(2*m0 - alpha*t1))
    else:
        v_anal1 = (vrel*alpha/gamma) * (1 - (1 - t1/(2*T))**(gamma/alpha))
    
    v1_handoff = v_anal1[-1]
    t2 = t_arr_r[t_arr_r > T]
    if gamma == 0:
        v_anal2 = np.full_like(t2, v1_handoff)
    else:
        v_anal2 = v1_handoff * np.exp(-gamma*(t2 - T)/m0)
    
    v_anal = np.concatenate([v_anal1, v_anal2])
    
    plt.plot(t_arr_r, v_arr_r, label=f'RK4 γ={gamma:.2f}')
    plt.plot(np.concatenate([t1, t2]), v_anal, '--', label=f'Analytic γ={gamma:.2f}')

plt.axvline(T, color='k', linestyle=':', label='End Phase 1')
plt.xlabel('t'); plt.ylabel('v')
plt.title('Rocket Velocity')
plt.legend(); plt.grid(); plt.tight_layout(); plt.show()

---
# ANALYTICAL CHEAT SHEET

## Terminal velocity problems:
- EOM: dv/dt = (vT - v)/τ
- v(t) = vT(1 - e^{-t/τ})
- t* = τ ln(100)  [time to reach 99% vT]
- x(t) = vT·[t + τ·e^{-t/τ} - τ]

## Rocket (variable mass):
- F_net = vrel·(dm/dt) - γv = -vrel·α - γv  (m burns at rate α)
- m(t)·dv/dt = vrel·α - γv
- Phase 2: m0·dv/dt = -γv → v = v1·e^{-γ(t-T)/m0}
- Tsiolkovsky (γ=0): v = vrel·ln(m_initial/m_final)

## SHM (for reference):
- y'' = -ω²y → y(t) = A cos(ωt) + B sin(ωt)
- For WS: ω = 2π, period = 1

## TISE bound states:
- κ = √(-E)  [decay constant for |x| > a]
- ψ ~ e^{-κ|x|} far from well
- n-th eigenfunction has n nodes
- Ground state (n=0) has 0 nodes, always symmetric for symmetric V

## Energy in damped system:
- E(t) = (1/2)mv² + V(x)
- dE/dt = -γv² ≤ 0  (always decreasing)
- Turning points: v=0, so E(t*) = V(x*)

## Phase portrait:
- Undamped: closed ellipses
- Damped: inward spiral → fixed point at (x_eq, 0)
- Fixed point = potential minimum (where dV/dx = 0)

---

## KEY THINGS THAT TRIP PEOPLE UP:
1. **Thomas algorithm modifies arrays** — always pass .copy()
2. **Simpson's 1/3 requires even N** (odd number of points)
3. **TDSE wavepacket needs exp(ik0x)** — otherwise it won't travel
4. **Dormand-Prince coefficients must not be modified**
5. **For shooting: start integration slightly inside the well** (at ±a + ε)
   because the potential has a vertical tangent at ±a for elliptic well
6. **Log-derivative matching removes normalization ambiguity**
7. **Interpolation needed** when bisecting on RK4 solution (it's discrete)
8. **Phase 1 → Phase 2 switch** in rocket: check `if t <= T`